In [ ]:
import sys
import os
sys.path.insert(0, '/app')

from datetime import datetime, date
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, DateType, TimestampType
from clickhouse_driver import Client
from connectors.clickhouse_client import ClickHouseClient
from config.settings import clickhouse_config


In [2]:
display_config = {
    "host": clickhouse_config.host,
    "port": clickhouse_config.port,
    "database": clickhouse_config.database,
    "secure": clickhouse_config.secure,
    "user": clickhouse_config.user,
}
display_config

{'host': 'e1a1lieug8.us-central1.gcp.clickhouse.cloud',
 'port': 8443,
 'database': 'default',
 'secure': True,
 'user': 'default'}

In [ ]:
import sys
import os
sys.path.insert(0, '/app')

from pyspark.sql import SparkSession
from config.settings import clickhouse_config

spark_home = os.environ.get('SPARK_HOME', '/usr/local/spark')
clickhouse_jar = f"{spark_home}/jars/clickhouse-jdbc-0.4.6-all.jar"

spark = (
    SparkSession.builder
    .appName("VerificacaoClickHouse")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.shuffle.targetPostShuffleInputSize", "64MB")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.jars", clickhouse_jar)
    .getOrCreate()
)

database = "siga"
table = "CN9030"
limit = 31
offset = 0

protocol = "https" if clickhouse_config.secure else "http"
clickhouse_jdbc_url = (
    f"jdbc:clickhouse://{protocol}://{clickhouse_config.host}:"
    f"{clickhouse_config.port}/{database}"
)

print("=" * 80)
print(f"LENDO TABELA CLICKHOUSE: {database}.{table}")
print("=" * 80)
print(f"JDBC URL: {clickhouse_jdbc_url}")
print(f"Database: {database}")
print(f"Table: {table}")
print(f"Limit: {limit}, Offset: {offset}")
print()

try:
    query = f"(SELECT * FROM {table} LIMIT {limit} OFFSET {offset})"
    
    df_bronze = (
        spark.read
        .format("jdbc")
        .option("url", clickhouse_jdbc_url)
        .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
        .option("dbtable", query)
        .option("user", clickhouse_config.user)
        .option("password", clickhouse_config.password)
        .option("batchsize", "50000")
        .load()
    )

    display(df_bronze)
    
    print(f"✓ Tabela lida com sucesso!")
    print(f"Total de registros: {df_bronze.count()}")
    print()
    print("Schema:")
    df_bronze.printSchema()
    print()
    print("Dados:")
    df_bronze.show(truncate=False)
    
except Exception as e:
    print(f"✗ Erro ao ler tabela: {str(e)}")
    print()
    print("Tentando variações...")
    
    variations = [
        (f"(SELECT * FROM `{table}` LIMIT {limit} OFFSET {offset})", f"{database}.{table}"),
        (f"(SELECT * FROM {database}.{table} LIMIT {limit} OFFSET {offset})", f"{database}.{table}"),
    ]
    
    for query_var, table_name in variations:
        try:
            df_bronze = (
                spark.read
                .format("jdbc")
                .option("url", clickhouse_jdbc_url)
                .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
                .option("dbtable", query_var)
                .option("user", clickhouse_config.user)
                .option("password", clickhouse_config.password)
                .option("batchsize", "50000")
                .load()
            )
            print(f"✓ Sucesso com query: {query_var}")
            print(f"Total de registros: {df_bronze.count()}")
            df_bronze.show(truncate=False)
            break
        except Exception as ve:
            print(f"✗ Falhou: {query_var}")
            print(f"  Erro: {str(ve)[:200]}")

from config.settings import clickhouse_config
from connectors.clickhouse_client import ClickHouseClient

print("=" * 80)
print("VERIFICAÇÃO DE CREDENCIAIS CLICKHOUSE")
print("=" * 80)
print(f"Host: {clickhouse_config.host}")
print(f"Port: {clickhouse_config.port}")
print(f"Database: {clickhouse_config.database}")
print(f"User: {clickhouse_config.user}")
print(f"Password: {'*' * len(clickhouse_config.password) if clickhouse_config.password else '(vazio)'}")
print(f"Secure: {clickhouse_config.secure}")
print(f"Verify: {clickhouse_config.verify}")
print()

print("Verificando variáveis de ambiente...")
clickhouse_env_vars = {k: v for k, v in os.environ.items() if k.startswith('CLICKHOUSE_')}
if clickhouse_env_vars:
    print(f"✓ {len(clickhouse_env_vars)} variáveis CLICKHOUSE encontradas no ambiente:")
    for key, value in sorted(clickhouse_env_vars.items()):
        if 'PASSWORD' in key:
            print(f"  {key}={'*' * len(value) if value else '(vazio)'}")
        else:
            print(f"  {key}={value}")
else:
    print("✗ Nenhuma variável CLICKHOUSE encontrada no ambiente")
    print("  O Docker Compose deve injetar as variáveis via env_file")

env_file_path = "/app/.env"
if os.path.exists(env_file_path):
    print(f"\n✓ Arquivo .env também encontrado em: {env_file_path}")
else:
    print(f"\nℹ Arquivo .env não encontrado em: {env_file_path}")
    print("  (Isso é normal se o Docker Compose usar env_file sem montar o arquivo)")

print()
print("=" * 80)
print("TENTANDO CONECTAR AO CLICKHOUSE...")
print("=" * 80)

try:
    clickhouse_client = ClickHouseClient()
    query_result = clickhouse_client.execute_query_with_result("SELECT 1 as value FROM system.one")
    records = query_result.result_set
    columns = query_result.column_names
    
    print("✓ Conexão estabelecida com sucesso!")
    print(f"Resultado da query: {records[0][0]}")
    
    if 'spark' in globals():
        df_clickhouse = spark.createDataFrame(records, schema=columns)
        df_clickhouse.show(truncate=False)
    
    clickhouse_client.close()
except Exception as e:
    print(f"✗ Erro ao conectar: {str(e)}")
    print()
    print("POSSÍVEIS SOLUÇÕES:")
    print("1. Verifique se as credenciais no arquivo .env estão corretas")
    print("2. Verifique se o arquivo .env está montado no container Docker")
    print("3. Para ClickHouse Cloud, redefina a senha em:")
    print("   https://clickhouse.cloud/")
    print("4. Verifique se o host e porta estão corretos")



LENDO TABELA CLICKHOUSE: siga.CN9030
JDBC URL: jdbc:clickhouse://https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443/siga
Database: siga
Table: CN9030
Limit: 31, Offset: 0



DataFrame[CN9_FILIAL: string, CN9_NUMERO: string, CN9_TPCTO: string, CN9_CLIENT: string, CN9_LOJACL: string, CN9_TWCNTE: string, CN9_DTINIC: string, CN9_DTASSI: string, CN9_VIGE: bigint, CN9_UNVIGE: string, CN9_DTFIM: string, CN9_TWMDAF: string, CN9_TWMDPS: string, CN9_CASTIG: string, CN9_CASTPR: bigint, CN9_TWRENA: string, CN9_TWFATD: string, CN9_TWRDV: string, CN9_TWRMAX: bigint, CN9_RRV: string, CN9_TWEQVE: bigint, CN9_MOEDA: bigint, CN9_CONDPG: string, CN9_CODOBJ: string, CN9_VLINI: double, CN9_VLATU: double, CN9_FLGREJ: string, CN9_INDICE: string, CN9_REVISA: string, CN9_FLGCAU: string, CN9_TPCAUC: string, CN9_MINCAU: bigint, CN9_DTENCE: string, CN9_TIPREV: string, CN9_REVATU: string, CN9_SALDO: double, CN9_MOTPAR: string, CN9_DTFIMP: string, CN9_DTREIN: string, CN9_CODJUS: string, CN9_CODCLA: string, CN9_DTREV: string, CN9_DTREAJ: string, CN9_VLREAJ: bigint, CN9_VLADIT: bigint, CN9_NUMTIT: string, CN9_VLMEAC: bigint, CN9_TXADM: bigint, CN9_FORMA: string, CN9_DTENTR: string, CN9_L

✓ Tabela lida com sucesso!
Total de registros: 31

Schema:
root
 |-- CN9_FILIAL: string (nullable = true)
 |-- CN9_NUMERO: string (nullable = true)
 |-- CN9_TPCTO: string (nullable = true)
 |-- CN9_CLIENT: string (nullable = true)
 |-- CN9_LOJACL: string (nullable = true)
 |-- CN9_TWCNTE: string (nullable = true)
 |-- CN9_DTINIC: string (nullable = true)
 |-- CN9_DTASSI: string (nullable = true)
 |-- CN9_VIGE: long (nullable = true)
 |-- CN9_UNVIGE: string (nullable = true)
 |-- CN9_DTFIM: string (nullable = true)
 |-- CN9_TWMDAF: string (nullable = true)
 |-- CN9_TWMDPS: string (nullable = true)
 |-- CN9_CASTIG: string (nullable = true)
 |-- CN9_CASTPR: long (nullable = true)
 |-- CN9_TWRENA: string (nullable = true)
 |-- CN9_TWFATD: string (nullable = true)
 |-- CN9_TWRDV: string (nullable = true)
 |-- CN9_TWRMAX: long (nullable = true)
 |-- CN9_RRV: string (nullable = true)
 |-- CN9_TWEQVE: long (nullable = true)
 |-- CN9_MOEDA: long (nullable = true)
 |-- CN9_CONDPG: string (nullab

In [4]:
import sys
sys.path.insert(0, '/app')

from connectors.clickhouse_client import ClickHouseClient
from config.settings import clickhouse_config

table_name = "CN9030"
limit = 31
offset = 0

print("=" * 80)
print(f"CONSULTA CLICKHOUSE: {table_name}")
print("=" * 80)
print(f"Database: {clickhouse_config.database}")
print(f"Tabela: {table_name}")
print(f"Limit: {limit}, Offset: {offset}")
print()

try:
    client = ClickHouseClient()
    
    query = f"SELECT * FROM {table_name} LIMIT {limit} OFFSET {offset}"
    print(f"Query: {query}")
    print()
    
    result = client.execute_query_with_result(query)
    
    if result.result_set:
        print(f"✓ {len(result.result_set)} registros encontrados")
        print()
        print("Colunas:", result.column_names)
        print()
        print("Dados:")
        for i, row in enumerate(result.result_set, 1):
            print(f"Linha {i}: {row}")
    else:
        print("Nenhum registro encontrado")
    
    client.close()
    
except Exception as e:
    print(f"✗ Erro ao executar query: {str(e)}")
    print()
    print("Tentando variações da query...")
    
    variations = [
        f"SELECT * FROM `{table_name}` LIMIT {limit} OFFSET {offset}",
        f"SELECT * FROM {clickhouse_config.database}.{table_name} LIMIT {limit} OFFSET {offset}",
        f"SELECT * FROM `{clickhouse_config.database}`.`{table_name}` LIMIT {limit} OFFSET {offset}",
    ]
    
    for var_query in variations:
        try:
            client = ClickHouseClient()
            result = client.execute_query_with_result(var_query)
            print(f"✓ Sucesso com: {var_query}")
            if result.result_set:
                print(f"  {len(result.result_set)} registros encontrados")
            client.close()
            break
        except Exception as ve:
            print(f"✗ Falhou: {var_query}")
            print(f"  Erro: {str(ve)[:100]}")


CONSULTA CLICKHOUSE: CN9030
Database: default
Tabela: CN9030
Limit: 31, Offset: 0

✗ Erro ao executar query: :HTTPDriver for https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443 returned response code 401)
 Code: 194. DB::Exception: default: Authentication failed: password is incorrect, or there is no user with such name.

If you use ClickHouse Cloud, the password can be reset at https://clickhouse.cloud/
on the settings page for the corresponding service.

If you have installed ClickHouse and forgot password you can reset it in the configuration file.
The password for default user is typically located at /etc/clickhouse-server/users.d/default-password.xml
and deleting this file will reset the password.
See also /etc/clickhouse-server/users.xml on the server where ClickHouse is installed.

. (REQUIRED_PASSWORD) (version 25.8.1.8702 (official build))


Tentando variações da query...
✗ Falhou: SELECT * FROM `CN9030` LIMIT 31 OFFSET 0
  Erro: :HTTPDriver for https://e1a1lieug8.us-cent

In [6]:
import sys
sys.path.insert(0, '/app')

from pyspark.sql import SparkSession
from config.settings import clickhouse_config

print("=" * 80)
print("DIAGNÓSTICO DE CREDENCIAIS CLICKHOUSE")
print("=" * 80)
print(f"Host: {clickhouse_config.host}")
print(f"Port: {clickhouse_config.port}")
print(f"Database: {clickhouse_config.database}")
print(f"User: {clickhouse_config.user}")
print(f"Password: {'*' * len(clickhouse_config.password) if clickhouse_config.password else '(vazio)'}")
print(f"Secure: {clickhouse_config.secure}")
print()

def load_cnb030_table():
    spark = SparkSession.builder.getOrCreate()
    
    database = "siga"
    table = "CNB030"
    
    if clickhouse_config.secure:
        protocol = "https"
        jdbc_url = f"jdbc:clickhouse://{protocol}://{clickhouse_config.host}:{clickhouse_config.port}/{database}?ssl=true&sslMode=strict"
    else:
        protocol = "http"
        jdbc_url = f"jdbc:clickhouse://{protocol}://{clickhouse_config.host}:{clickhouse_config.port}/{database}"
    
    print(f"JDBC URL: {jdbc_url}")
    print(f"Tabela: {database}.{table}")
    print()
    
    try:
        query = f"(SELECT * FROM {table} LIMIT 100)"
        
        df = (
            spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
            .option("user", clickhouse_config.user)
            .option("password", clickhouse_config.password)
            .option("dbtable", query)
            .option("batchsize", "50000")
            .load()
        )
        
        print(f"✓ Tabela carregada com sucesso!")
        print(f"Total de registros: {df.count()}")
        print()
        df.printSchema()
        print()
        df.show(truncate=False)
        
        return df
        
    except Exception as e:
        error_msg = str(e)
        print(f"✗ Erro ao carregar tabela: {error_msg[:500]}")
        print()
        
        if "Authentication failed" in error_msg or "REQUIRED_PASSWORD" in error_msg:
            print("ERRO DE AUTENTICAÇÃO:")
            print("1. Verifique se a senha no arquivo .env está correta")
            print("2. Para ClickHouse Cloud, redefina a senha em:")
            print("   https://clickhouse.cloud/")
            print("3. Reinicie o container após atualizar o .env:")
            print("   docker-compose restart jupyter")
        
        raise

display(load_cnb030_table())





DIAGNÓSTICO DE CREDENCIAIS CLICKHOUSE
Host: e1a1lieug8.us-central1.gcp.clickhouse.cloud
Port: 8443
Database: default
User: default
Password: *************
Secure: True

JDBC URL: jdbc:clickhouse://https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443/siga?ssl=true&sslMode=strict
Tabela: siga.CNB030

✗ Erro ao carregar tabela: An error occurred while calling o76.load.
: java.sql.SQLException: Code: 194. DB::Exception: default: Authentication failed: password is incorrect, or there is no user with such name.

If you use ClickHouse Cloud, the password can be reset at https://clickhouse.cloud/
on the settings page for the corresponding service.

If you have installed ClickHouse and forgot password you can reset it in the configuration file.
The password for default user is typically located at /etc/clickhouse-server/user

ERRO DE AUTENTICAÇÃO:
1. Verifique se a senha no arquivo .env está correta
2. Para ClickHouse Cloud, redefina a senha em:
   https://clickhouse.cloud/
3. Reinicie o co

Py4JJavaError: An error occurred while calling o76.load.
: java.sql.SQLException: Code: 194. DB::Exception: default: Authentication failed: password is incorrect, or there is no user with such name.

If you use ClickHouse Cloud, the password can be reset at https://clickhouse.cloud/
on the settings page for the corresponding service.

If you have installed ClickHouse and forgot password you can reset it in the configuration file.
The password for default user is typically located at /etc/clickhouse-server/users.d/default-password.xml
and deleting this file will reset the password.
See also /etc/clickhouse-server/users.xml on the server where ClickHouse is installed.

. (REQUIRED_PASSWORD) (version 25.8.1.8702 (official build))
, server ClickHouseNode [uri=https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443/siga, options={sslMode=strict,sslmode=STRICT}]@-295446472
	at com.clickhouse.jdbc.SqlExceptionUtils.handle(SqlExceptionUtils.java:85)
	at com.clickhouse.jdbc.SqlExceptionUtils.c

In [27]:
from pyspark.sql import SparkSession
from config.settings import clickhouse_config

jdbc_url = f"jdbc:clickhouse://{clickhouse_config.host}:{clickhouse_config.port}/{clickhouse_config.database}?ssl=true&sslMode=strict"

df_clickhouse_jdbc = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
    .option("user", clickhouse_config.user)
    .option("password", clickhouse_config.password)
    .option("query", "SELECT 1 as value FROM system.one")
    .load()
)

df_clickhouse_jdbc.show(truncate=False)



Py4JJavaError: An error occurred while calling o200.load.
: java.sql.SQLException: Code: 194. DB::Exception: default: Authentication failed: password is incorrect, or there is no user with such name.

If you use ClickHouse Cloud, the password can be reset at https://clickhouse.cloud/
on the settings page for the corresponding service.

If you have installed ClickHouse and forgot password you can reset it in the configuration file.
The password for default user is typically located at /etc/clickhouse-server/users.d/default-password.xml
and deleting this file will reset the password.
See also /etc/clickhouse-server/users.xml on the server where ClickHouse is installed.

. (REQUIRED_PASSWORD) (version 25.8.1.8702 (official build))
, server ClickHouseNode [uri=https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443/default, options={sslMode=strict}]@1162128802
	at com.clickhouse.jdbc.SqlExceptionUtils.handle(SqlExceptionUtils.java:85)
	at com.clickhouse.jdbc.SqlExceptionUtils.create(SqlEx

In [10]:
clickhouse_client = ClickHouseClient()
result = clickhouse_client.execute_query_with_result("SELECT 1 as value FROM system.one")
print(f"Resultado direto do ClickHouse: {result.result_set[0][0]}")
clickhouse_client.close()



DatabaseError: :HTTPDriver for https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443 returned response code 401)
 Code: 194. DB::Exception: default: Authentication failed: password is incorrect, or there is no user with such name.

If you use ClickHouse Cloud, the password can be reset at https://clickhouse.cloud/
on the settings page for the corresponding service.

If you have installed ClickHouse and forgot password you can reset it in the configuration file.
The password for default user is typically located at /etc/clickhouse-server/users.d/default-password.xml
and deleting this file will reset the password.
See also /etc/clickhouse-server/users.xml on the server where ClickHouse is installed.

. (REQUIRED_PASSWORD) (version 25.8.1.8702 (official build))


In [11]:
# Configuration
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
CLICKHOUSE_HOST = "clickhouse"
CLICKHOUSE_PORT = 9000
CLICKHOUSE_USER = "default"
CLICKHOUSE_PASSWORD = ""
CLICKHOUSE_DB = "bronze"


In [12]:
# Spark Session
spark = (SparkSession.builder
    .appName("OracleToClickHouseIngestion")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate())


In [13]:
spark.sql("SELECT 1 FROM clickhouse.`system`.`one`").show()

Py4JError: An error occurred while calling o24.sql. Trace:
py4j.Py4JException: Method sql([class java.lang.String, class [Ljava.lang.Object;]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:321)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:329)
	at py4j.Gateway.invoke(Gateway.java:274)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.base/java.lang.Thread.run(Thread.java:833)



In [5]:
# Schema Definition
schema_oracle = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("amount", DecimalType(18, 2), True),
    StructField("transaction_date", DateType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("status", StringType(), True)
])


In [14]:
# Definição dos parâmetros de conexão com base no contexto (.env)
oracle_host = "10.255.150.11"
oracle_port = "1521"
oracle_service = "bi.grupotracker.com.br"
oracle_user = "clickhouse"
oracle_password = "qiU!EOoe"

jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service}"
print(f"Iniciando teste de conexão com: {jdbc_url}")

try:
    # Teste simples usando a tabela DUAL do Oracle
    # A query valida se conseguimos executar SQL no banco
    test_query = "(SELECT 'Conexão OK' as status, sysdate as data_hora FROM dual)"

    df_test_oracle = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("dbtable", test_query)
        .option("user", oracle_user)
        .option("password", oracle_password)
        .load()
    )

    print("Schema detectado:")
    df_test_oracle.printSchema()

    print("Resultado da consulta:")
    df_test_oracle.show(truncate=False)

except Exception as e:
    print(f"Erro ao conectar no Oracle: {str(e)}")


Iniciando teste de conexão com: jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# Consulta para listar todos os schemas (owner) e tabelas acessíveis
query_metadata = "(SELECT owner, table_name FROM all_tables ORDER BY owner, table_name)"

try:
    print("Lendo estrutura de tabelas e schemas do Oracle...")
    df_structure = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("dbtable", query_metadata)
        .option("user", oracle_user)
        .option("password", oracle_password)
        .load()
    )

    print("Amostra de tabelas encontradas:")
    df_structure.show(10, truncate=False)

    print("Contagem de tabelas por Schema:")
    df_structure.groupBy("owner").count().orderBy(F.col("count").desc()).show()

except Exception as e:
    print(f"Erro ao ler metadados: {str(e)}")


Lendo estrutura de tabelas e schemas do Oracle...


In [7]:
# Instalação do clickhouse-connect (se necessário)
try:
    import clickhouse_connect
except ImportError:
    !pip install clickhouse-connect
    import clickhouse_connect

from pyspark.sql.functions import current_timestamp

# 1. Lista todos os schemas e tabelas disponíveis no Oracle para gerar ingestion dinâmica
schemas_tables_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("dbtable", "(SELECT owner, table_name FROM all_tables ORDER BY owner, table_name)")
    .option("user", oracle_user)
    .option("password", oracle_password)
    .load()
    .select(
        F.col("OWNER").alias("owner"),
        F.col("TABLE_NAME").alias("table_name"),
    )
)

display(schemas_tables_df)

schemas_tables = schemas_tables_df.collect()

# 2. Define um modelo/padrão de mapeamento (colunas destino em ClickHouse)
clickhouse_columns = [
    ("id", "UInt64"),
    ("nome", "String"),
    ("valor", "Float64"),
    ("importado_em", "DateTime"),
]


def get_mapped_df(df):
    return (
        df
        .withColumn("importado_em", current_timestamp())
        .withColumnRenamed("transaction_id", "id")
        .withColumnRenamed("customer_id", "nome")
        .withColumnRenamed("amount", "valor")
        .select("id", "nome", "valor", "importado_em")
    )


def generate_ddl(database, table, columns):
    cols = ",\n    ".join([f"{col} {dtype}" for col, dtype in columns])
    return f"""
        CREATE TABLE IF NOT EXISTS {database}.{table}
        (
            {cols}
        ) ENGINE = MergeTree()
        ORDER BY id
    """


if __name__ == '__main__':
    client = clickhouse_connect.get_client(
        host='fxo48y5409.us-east1.gcp.clickhouse.cloud',
        user='default',
        password='1cb92kn~fcqO7',
        secure=True
    )
    client.command("CREATE DATABASE IF NOT EXISTS stage_demo")

    for row in schemas_tables:
        print(row.asDict())  # Adicione esta linha para inspecionar quais campos existem
        # continue com o uso correto do campo...
        schema = row['owner']
        table = row['table_name']
        oracle_full_table = f"{schema}.{table}"

        try:
            # UTILIZANDO O SPARK PARA LER DO ORACLE
            df_oracle = (
                spark.read.format("jdbc")
                .option("url", jdbc_url)
                .option("driver", "oracle.jdbc.OracleDriver")
                .option("dbtable", oracle_full_table)
                .option("user", oracle_user)
                .option("password", oracle_password)
                .load()
            )

            # Checa se as colunas necessárias existem antes de mapear
            required_cols = {'transaction_id', 'customer_id', 'amount'}
            if not required_cols.issubset(set(df_oracle.columns)):
                print(f"Pulando {oracle_full_table}: colunas obrigatórias não encontradas")
                continue

            # PROCESSAMENTO E MAPEAMENTO VIA SPARK
            df_clickhouse = get_mapped_df(df_oracle)
            stage_table = f"staging_{schema.lower()}_{table.lower()}"
            ddl = generate_ddl("stage_demo", stage_table, clickhouse_columns)
            client.command(ddl)
            print(f"Tabela {stage_table} criada/valida no ClickHouse.")

            # ENVIANDO OS DADOS VIA SPARK, CASO NECESSÁRIO USE SPARK-TURN CLICKHOUSE JDBC CONN
            # Aqui converte para pandas apenas para exemplo prático simples, mas normalmente deveria ser .write.format("jdbc")
            data_to_insert = df_clickhouse.toPandas().itertuples(index=False, name=None)
            insert_query = f"INSERT INTO stage_demo.{stage_table} (id, nome, valor, importado_em) VALUES"
            client.insert(insert_query, list(data_to_insert))
            print(f"Dados de {oracle_full_table} enviados para {stage_table}!")

            row_count = client.query(f"SELECT count(*) FROM stage_demo.{stage_table}").result_set[0][0]
            print(f"{stage_table}: {row_count} linhas no ClickHouse.")

        except Exception as e:
            print(f"Erro processando {oracle_full_table}: {str(e)}")


Py4JJavaError: An error occurred while calling o60.load.
: java.sql.SQLRecoverableException: IO Error: The Network Adapter could not establish the connection (CONNECTION_ID=0URUQCJSRyWmSdjOPw8DZg==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CConnection.logon(T4CConnection.java:697)
	at oracle.jdbc.driver.PhysicalConnection.connect(PhysicalConnection.java:1047)
	at oracle.jdbc.driver.T4CDriverExtension.getConnection(T4CDriverExtension.java:89)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:732)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:648)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1(JdbcDialects.sca

In [ ]:
# Staging (Simulating Write to Shared Storage/Object Store)
staging_path = "/app/temp/staging_transactions"
df_oracle.write.mode("overwrite").parquet(staging_path)


In [14]:
# ClickHouse Connection
client = Client(host=CLICKHOUSE_HOST, port=CLICKHOUSE_PORT, user=CLICKHOUSE_USER, password=CLICKHOUSE_PASSWORD)


In [ ]:
# DDL Execution (Bronze Layer)
client.execute(f"CREATE DATABASE IF NOT EXISTS {CLICKHOUSE_DB}")

ddl_bronze = f"""
CREATE TABLE IF NOT EXISTS {CLICKHOUSE_DB}.transactions_local
(
    transaction_id String,
    customer_id UInt32,
    amount Decimal(18,2),
    transaction_date Date,
    created_at DateTime,
    status String,
    ingestion_date Date DEFAULT today()
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(transaction_date)
ORDER BY (transaction_date, customer_id, transaction_id)
"""
client.execute(ddl_bronze)


In [ ]:
# Ingestion from Staging to ClickHouse
# Reading back from Parquet to simulate the decoupling
df_staging = spark.read.parquet(staging_path)

# Convert to list of tuples for ClickHouse driver
# For massive datasets, we would use clickhouse-client via subprocess or specialized format writer
data_to_insert = df_staging.collect()

insert_query = f"INSERT INTO {CLICKHOUSE_DB}.transactions_local (transaction_id, customer_id, amount, transaction_date, created_at, status) VALUES"
client.execute(insert_query, data_to_insert)


In [ ]:
# Validation
result = client.execute(f"SELECT count(), sum(amount) FROM {CLICKHOUSE_DB}.transactions_local")
print(f"Rows inserted: {result[0][0]}")
print(f"Total Amount: {result[0][1]}")

packages = [
    "com.clickhouse.spark:clickhouse-spark-runtime-3.4_2.12:0.8.0",
    "com.clickhouse:clickhouse-client:0.7.0",
    "com.clickhouse:clickhouse-http-client:0.7.0",
    "com.oracle.database.jdbc:ojdbc8:23.2.0.0",
    "org.apache.httpcomponents.client5:httpclient5:5.2.1"
]

spark = (
    SparkSession.builder
        .appName("OracleToClickHouse")
        .master("spark://spark-master:7077")
        .config("spark.jars.packages", ",".join(packages))
        .config("spark.sql.shuffle.partitions", "4")
        .getOrCreate()
)

In [ ]:
oracle_df = (
    spark.read.format("jdbc")
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("url", "jdbc:oracle:thin:@//ORACLE_HOST:1521/SERVICE_NAME")
    .option("user", "ORACLE_USER")
    .option("password", "ORACLE_PASS")
    .option("dbtable", "SCHEMA.TABELA_FONTE")
    .load()
)

oracle_df.show(5)

In [8]:
query = (
"SELECT 1 FROM clickhouse.`system`.`one`"
)
result = client.execute(query)
for row in result:
    print(row)



NameError: name 'client' is not defined

In [13]:
spark.sql("SELECT 1 FROM clickhouse.`system`.`one`").show()



Py4JError: An error occurred while calling o31.sql. Trace:
py4j.Py4JException: Method sql([class java.lang.String, class [Ljava.lang.Object;]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:321)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:329)
	at py4j.Gateway.invoke(Gateway.java:274)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.base/java.lang.Thread.run(Thread.java:833)



In [3]:
import sys
import socket
sys.path.insert(0, '/app')

from config.settings import oracle_config

print("=" * 80)
print("DIAGNÓSTICO DE CONECTIVIDADE DE REDE - ORACLE")
print("=" * 80)
print(f"Host configurado: {oracle_config.host}")
print(f"Porta configurada: {oracle_config.port}")
print()

try:
    ip_address = socket.gethostbyname(oracle_config.host)
    print(f"✓ Resolução DNS: {oracle_config.host} -> {ip_address}")
except socket.gaierror as e:
    print(f"✗ Erro na resolução DNS: {e}")
    ip_address = oracle_config.host

print()
print("Testando conectividade TCP...")
try:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(5)
    result = sock.connect_ex((ip_address, int(oracle_config.port)))
    sock.close()
    
    if result == 0:
        print(f"✓ Porta {oracle_config.port} está acessível em {ip_address}")
    else:
        print(f"✗ Porta {oracle_config.port} não está acessível em {ip_address} (código: {result})")
        print("  Possíveis causas:")
        print("  - Firewall bloqueando a conexão")
        print("  - Servidor Oracle não está rodando")
        print("  - Rede não acessível do container Docker")
except socket.timeout:
    print(f"✗ Timeout ao tentar conectar em {ip_address}:{oracle_config.port}")
except Exception as e:
    print(f"✗ Erro ao testar conectividade: {e}")

print()
print("=" * 80)


DIAGNÓSTICO DE CONECTIVIDADE DE REDE - ORACLE
Host configurado: 10.255.150.11
Porta configurada: 1521

✓ Resolução DNS: 10.255.150.11 -> 10.255.150.11

Testando conectividade TCP...
✗ Porta 1521 não está acessível em 10.255.150.11 (código: 11)
  Possíveis causas:
  - Firewall bloqueando a conexão
  - Servidor Oracle não está rodando
  - Rede não acessível do container Docker



In [2]:
import sys
sys.path.insert(0, '/app')

from pyspark.sql import SparkSession
from config.settings import oracle_config, clickhouse_config
from connectors.clickhouse_client import ClickHouseClient

print("=" * 80)
print("CONFIGURAÇÃO DE CONEXÃO ORACLE")
print("=" * 80)
print(f"Host: {oracle_config.host}")
print(f"Port: {oracle_config.port}")
print(f"Service: {oracle_config.service}")
print(f"User: {oracle_config.user}")
print(f"Password: {'*' * len(oracle_config.password)}")
print()

oracle_jdbc_url = f"jdbc:oracle:thin:@//{oracle_config.host}:{oracle_config.port}/{oracle_config.service}"
print(f"JDBC URL: {oracle_jdbc_url}")
print()

spark = SparkSession.builder.appName("ConnectionTest").getOrCreate()

try:
    test_query = "(SELECT 'Conexão Oracle OK' as status, sysdate as data_hora FROM dual)"
    df_test = (
        spark.read.format("jdbc")
        .option("url", oracle_jdbc_url)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("dbtable", test_query)
        .option("user", oracle_config.user)
        .option("password", oracle_config.password)
        .load()
    )
    print("✓ Conexão Oracle estabelecida com sucesso!")
    df_test.show(truncate=False)
except Exception as e:
    print(f"✗ Erro ao conectar no Oracle: {str(e)}")


CONFIGURAÇÃO DE CONEXÃO ORACLE
Host: 10.255.150.11
Port: 1521
Service: bi.grupotracker.com.br
User: clickhouse
Password: ********

JDBC URL: jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br

✗ Erro ao conectar no Oracle: An error occurred while calling o32.load.
: java.sql.SQLRecoverableException: IO Error: The Network Adapter could not establish the connection (CONNECTION_ID=GvBBSnT/QwmdUM5V+/lohQ==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CConnection.logon(T4CConnection.java:697)
	at oracle.jdbc.driver.PhysicalConnection.connect(PhysicalConnection.java:1047)
	at oracle.jdbc.driver.T4CDriverExtension.getConnection(T4CDriverExtension.java:89)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:732)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:648)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:4

In [11]:
import sys
import socket
sys.path.insert(0, '/app')

from config.settings import oracle_config

print("=" * 80)
print("DIAGNÓSTICO DE CONECTIVIDADE ORACLE")
print("=" * 80)
print(f"Host: {oracle_config.host}")
print(f"Port: {oracle_config.port}")
print(f"Service: {oracle_config.service}")
print(f"User: {oracle_config.user}")
print()

try:
    ip_address = socket.gethostbyname(oracle_config.host)
    print(f"✓ Resolução DNS: {oracle_config.host} -> {ip_address}")
except socket.gaierror as e:
    print(f"✗ Erro na resolução DNS: {e}")
    ip_address = oracle_config.host

print()
print("Testando conectividade TCP...")
try:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(5)
    result = sock.connect_ex((ip_address, int(oracle_config.port)))
    sock.close()
    
    if result == 0:
        print(f"✓ Porta {oracle_config.port} está acessível em {ip_address}")
        print("  Conexão de rede OK - pode prosseguir com JDBC")
    else:
        print(f"✗ Porta {oracle_config.port} NÃO está acessível em {ip_address} (código: {result})")
        print()
        print("⚠ PROBLEMA DE CONECTIVIDADE IDENTIFICADO:")
        print("  O container Docker não consegue alcançar o servidor Oracle.")
        print()
        print("  POSSÍVEIS CAUSAS:")
        print("  1. Servidor Oracle não está rodando ou inacessível")
        print("  2. Firewall bloqueando a porta 1521")
        print("  3. Container Docker em rede diferente do servidor Oracle")
        print("  4. IP/hostname incorreto")
        print()
        print("  SOLUÇÕES:")
        print("  1. Verifique se o Oracle está rodando: telnet {ip_address} {oracle_config.port}")
        print("  2. Configure network_mode: 'host' no docker-compose.yml (Linux apenas)")
        print("  3. Use VPN/túnel se Oracle está em rede privada")
        print("  4. Verifique regras de firewall no servidor Oracle")
        print("  5. Teste conectividade do host (fora do Docker)")
except socket.timeout:
    print(f"✗ Timeout ao tentar conectar em {ip_address}:{oracle_config.port}")
    print("  O servidor pode estar inacessível ou a porta bloqueada")
except Exception as e:
    print(f"✗ Erro ao testar conectividade: {e}")

print()
print("=" * 80)


DIAGNÓSTICO DE CONECTIVIDADE ORACLE
Host: 10.255.150.11
Port: 1521
Service: bi.grupotracker.com.br
User: clickhouse

✓ Resolução DNS: 10.255.150.11 -> 10.255.150.11

Testando conectividade TCP...
✗ Porta 1521 NÃO está acessível em 10.255.150.11 (código: 11)

⚠ PROBLEMA DE CONECTIVIDADE IDENTIFICADO:
  O container Docker não consegue alcançar o servidor Oracle.

  POSSÍVEIS CAUSAS:
  1. Servidor Oracle não está rodando ou inacessível
  2. Firewall bloqueando a porta 1521
  3. Container Docker em rede diferente do servidor Oracle
  4. IP/hostname incorreto

  SOLUÇÕES:
  1. Verifique se o Oracle está rodando: telnet {ip_address} {oracle_config.port}
  2. Configure network_mode: 'host' no docker-compose.yml (Linux apenas)
  3. Use VPN/túnel se Oracle está em rede privada
  4. Verifique regras de firewall no servidor Oracle
  5. Teste conectividade do host (fora do Docker)



In [17]:
import sys
sys.path.insert(0, '/app')

from config.settings import clickhouse_config
from connectors.clickhouse_client import ClickHouseClient

print("=" * 80)
print("CONFIGURAÇÃO DE CONEXÃO CLICKHOUSE")
print("=" * 80)
print(f"Host: {clickhouse_config.host}")
print(f"Port: {clickhouse_config.port}")
print(f"Database: {clickhouse_config.database}")
print(f"User: {clickhouse_config.user}")
print(f"Password: {'*' * len(clickhouse_config.password)}")
print(f"Secure: {clickhouse_config.secure}")
print(f"Verify: {clickhouse_config.verify}")
print()

try:
    clickhouse_client = ClickHouseClient()
    result = clickhouse_client.execute_query_with_result("SELECT 'Conexão ClickHouse OK' as status, now() as data_hora")
    print("✓ Conexão ClickHouse estabelecida com sucesso!")
    for row in result.result_set:
        print(f"Status: {row[0]}, Data/Hora: {row[1]}")
    clickhouse_client.close()
except Exception as e:
    print(f"✗ Erro ao conectar no ClickHouse: {str(e)}")


CONFIGURAÇÃO DE CONEXÃO CLICKHOUSE
Host: e1a1lieug8.us-central1.gcp.clickhouse.cloud
Port: 8443
Database: default
User: default
Password: *************
Secure: True
Verify: True

✗ Erro ao conectar no ClickHouse: :HTTPDriver for https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443 returned response code 401)
 Code: 194. DB::Exception: default: Authentication failed: password is incorrect, or there is no user with such name.

If you use ClickHouse Cloud, the password can be reset at https://clickhouse.cloud/
on the settings page for the corresponding service.

If you have installed ClickHouse and forgot password you can reset it in the configuration file.
The password for default user is typically located at /etc/clickhouse-server/users.d/default-password.xml
and deleting this file will reset the password.
See also /etc/clickhouse-server/users.xml on the server where ClickHouse is installed.

. (REQUIRED_PASSWORD) (version 25.8.1.8702 (official build))



In [7]:
import sys
sys.path.insert(0, '/app')

from pyspark.sql import SparkSession
from config.settings import oracle_config

schemas_to_extract = [
    "ADMBI_PRD",
    "ALLSYSTEM_PROD",
    "BISTAGE",
    "CCTRACKERAVV",
    "CLICKHOUSE",
    "COMANDO",
    "CONSULTA",
    "DB_TESTE",
    "DIP",
    "DOUGLAS_FERREIRA",
    "DVF",
    "FERNANDO_RIBEIRO",
    "GINF",
    "HORUS",
    "HORUS_STAT",
    "IMPORT_USER",
    "LIVIADCORTE",
    "MDDATA",
    "MILLENA_PIVATO",
    "MPORTAL",
    "MULTIPORTAL",
    "NEXTAGE_PROD",
    "PDBUSER",
    "PERFSTAT",
    "RONALDO_ARIMURA",
    "SCOT",
    "SCOTGI",
    "SCOTSEQ",
    "SCOTT",
    "SCOT_BKP",
    "SCTHORUS",
    "SGTI",
    "SGTIGI",
    "SIGA",
    "SIGAGI",
    "SIGA_RO",
    "SUPORTEBD",
    "THIAGO_FERNANDES",
    "N8N_SERASA"
]

print("=" * 80)
print("SCHEMAS ORACLE PARA EXTRAÇÃO")
print("=" * 80)
print(f"Total de schemas: {len(schemas_to_extract)}")
print()

oracle_jdbc_url = f"jdbc:oracle:thin:@//{oracle_config.host}:{oracle_config.port}/{oracle_config.service}"
spark = SparkSession.builder.appName("SchemaList").getOrCreate()

try:
    query_schemas = f"""
    (SELECT DISTINCT owner as schema_name 
     FROM all_tables 
     WHERE owner IN ({','.join([f"'{s}'" for s in schemas_to_extract])})
     ORDER BY owner)
    """
    
    df_schemas = (
        spark.read.format("jdbc")
        .option("url", oracle_jdbc_url)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("dbtable", query_schemas)
        .option("user", oracle_config.user)
        .option("password", oracle_config.password)
        .load()
    )
    
    print("Schemas encontrados no Oracle:")
    df_schemas.show(truncate=False, n=100)
    
    schemas_found = [row.SCHEMA_NAME for row in df_schemas.collect()]
    schemas_not_found = [s for s in schemas_to_extract if s not in schemas_found]
    
    if schemas_not_found:
        print(f"\nSchemas não encontrados ({len(schemas_not_found)}):")
        for schema in schemas_not_found:
            print(f"  - {schema}")
    
    print(f"\n✓ Total de schemas válidos para extração: {len(schemas_found)}")
    
except Exception as e:
    print(f"✗ Erro ao listar schemas: {str(e)}")


SCHEMAS ORACLE PARA EXTRAÇÃO
Total de schemas: 39

✗ Erro ao listar schemas: An error occurred while calling o186.load.
: java.sql.SQLRecoverableException: IO Error: The Network Adapter could not establish the connection (CONNECTION_ID=Y5gsb+OoQB2iMWaFTevzvw==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CConnection.logon(T4CConnection.java:697)
	at oracle.jdbc.driver.PhysicalConnection.connect(PhysicalConnection.java:1047)
	at oracle.jdbc.driver.T4CDriverExtension.getConnection(T4CDriverExtension.java:89)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:732)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:648)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.

In [1]:
import sys
import os
sys.path.insert(0, '/app')

from config.settings import clickhouse_config

print("=" * 80)
print("DIAGNÓSTICO COMPLETO DE AUTENTICAÇÃO CLICKHOUSE")
print("=" * 80)
print()

print("1. VARIÁVEIS DE AMBIENTE DO SISTEMA:")
print("-" * 80)
clickhouse_env_vars = {k: v for k, v in os.environ.items() if k.startswith('CLICKHOUSE_')}
if clickhouse_env_vars:
    for key, value in sorted(clickhouse_env_vars.items()):
        if 'PASSWORD' in key:
            print(f"  {key}={'*' * len(value) if value else '(vazio)'} (tamanho: {len(value)})")
        else:
            print(f"  {key}={value}")
else:
    print("  ✗ Nenhuma variável CLICKHOUSE encontrada no ambiente")
print()

print("2. CONFIGURAÇÃO CARREGADA PELO PYDANTIC:")
print("-" * 80)
print(f"  Host: {clickhouse_config.host}")
print(f"  Port: {clickhouse_config.port}")
print(f"  Database: {clickhouse_config.database}")
print(f"  User: {clickhouse_config.user}")
print(f"  Password: {'*' * len(clickhouse_config.password) if clickhouse_config.password else '(vazio)'} (tamanho: {len(clickhouse_config.password) if clickhouse_config.password else 0})")
print(f"  Secure: {clickhouse_config.secure}")
print(f"  Verify: {clickhouse_config.verify}")
print()

print("3. VERIFICAÇÃO DO ARQUIVO .ENV:")
print("-" * 80)
env_file_paths = ["/app/.env", ".env", "/app/config/.env"]
env_found = False
for env_path in env_file_paths:
    if os.path.exists(env_path):
        print(f"  ✓ Arquivo encontrado: {env_path}")
        env_found = True
        try:
            with open(env_path, 'r') as f:
                lines = f.readlines()
                clickhouse_lines = [l.strip() for l in lines if l.strip().startswith('CLICKHOUSE_') and not l.strip().startswith('#')]
                if clickhouse_lines:
                    print(f"  Variáveis CLICKHOUSE no arquivo:")
                    for line in clickhouse_lines:
                        if 'PASSWORD' in line:
                            parts = line.split('=', 1)
                            if len(parts) == 2:
                                pwd = parts[1].strip().strip('"').strip("'")
                                print(f"    {parts[0]}={'*' * len(pwd)} (tamanho: {len(pwd)})")
                            else:
                                print(f"    {line}")
                        else:
                            print(f"    {line}")
                else:
                    print(f"  ✗ Nenhuma variável CLICKHOUSE encontrada no arquivo")
        except Exception as e:
            print(f"  ✗ Erro ao ler arquivo: {e}")
        break

if not env_found:
    print("  ✗ Arquivo .env não encontrado em nenhum dos caminhos testados")
    print("  Caminhos testados:", env_file_paths)
print()

print("4. TESTE DE CONEXÃO:")
print("-" * 80)
try:
    from connectors.clickhouse_client import ClickHouseClient
    client = ClickHouseClient()
    result = client.execute_query_with_result("SELECT 1 as test")
    print("  ✓ Conexão estabelecida com sucesso!")
    client.close()
except Exception as e:
    error_msg = str(e)
    print(f"  ✗ Erro de conexão: {error_msg[:300]}")
    print()
    if "Authentication failed" in error_msg or "REQUIRED_PASSWORD" in error_msg or "401" in error_msg:
        print("  ⚠ PROBLEMA IDENTIFICADO: Autenticação falhou")
        print()
        print("  SOLUÇÃO:")
        print("  1. Verifique a senha no arquivo .env (na raiz do projeto)")
        print("  2. Para ClickHouse Cloud, obtenha a senha correta em:")
        print("     https://clickhouse.cloud/")
        print("  3. Certifique-se de que a variável CLICKHOUSE_PASSWORD está correta")
        print("  4. Reinicie o container após atualizar:")
        print("     docker-compose restart jupyter")
        print("  5. Execute esta célula novamente")
print()
print("=" * 80)


DIAGNÓSTICO COMPLETO DE AUTENTICAÇÃO CLICKHOUSE

1. VARIÁVEIS DE AMBIENTE DO SISTEMA:
--------------------------------------------------------------------------------
  CLICKHOUSE_DATABASE=default
  CLICKHOUSE_HOST=e1a1lieug8.us-central1.gcp.clickhouse.cloud
  CLICKHOUSE_PASSWORD=************* (tamanho: 13)
  CLICKHOUSE_PORT=8443
  CLICKHOUSE_SECURE=true
  CLICKHOUSE_USER=default
  CLICKHOUSE_VERIFY=true

2. CONFIGURAÇÃO CARREGADA PELO PYDANTIC:
--------------------------------------------------------------------------------
  Host: e1a1lieug8.us-central1.gcp.clickhouse.cloud
  Port: 8443
  Database: default
  User: default
  Password: ************* (tamanho: 13)
  Secure: True
  Verify: True

3. VERIFICAÇÃO DO ARQUIVO .ENV:
--------------------------------------------------------------------------------
  ✗ Arquivo .env não encontrado em nenhum dos caminhos testados
  Caminhos testados: ['/app/.env', '.env', '/app/config/.env']

4. TESTE DE CONEXÃO:
-----------------------------------